# Training-free guidance (TFG) on DeepFashion images

Comparison method 2 of 3 on Modality 1: the condition enters through the **sampling
trajectory**. The diffusion model runs unconditionally and is steered at every step by the
gradient of a classifier evaluated on the predicted clean image. Nothing in the generative model
is trained: the diffusion model is the class-conditional base shared by every DeepFashion
notebook, run with its null label, and the classifier is a ResNet-34 fine-tuned once on the real
training images, because no pretrained DeepFashion classifier exists.

**Paper:** Ye, Lin, Han, Xu, Liu, Liang, Ma, Zou, Ermon.
*TFG: Unified Training-Free Guidance for Diffusion Models.* NeurIPS 2024.
[arXiv:2409.15761](https://arxiv.org/abs/2409.15761)

**Official code exists** and was read:
[YWolfeee/Training-Free-Guidance](https://github.com/YWolfeee/Training-Free-Guidance),
`methods/tfg.py`. `src/guidance/baselines/tfg.py` re-expresses `guide_step` against our
DDIM sampler, with their schedules and their CIFAR-10 label-guidance hyperparameters as
defaults. Its header lists what was kept and what changed.

**Two different classifiers.** A fine-tuned ResNet-34 guides; the fine-tuned ResNet-18 that
every DeepFashion notebook scores with evaluates. If one network did both, condition accuracy
would only measure how well guidance climbs its own objective.

**The target is the class.** TFG needs a differentiable objective on clean images; a classifier
gives one per category, not per caption, so this arm is conditioned on the 50-way label and
compares against the label-conditioned base. The text arms condition on captions; the scorer,
sampler, base weights and evaluation set are the same, so the accuracy and FID columns compare.

Run top to bottom. On Colab set **Runtime > Change runtime type > GPU** first. Run
`cfg_fashion.ipynb` first, or place its `checkpoints/diffusion_fashion/best.pt` next to this
notebook, and base training is skipped.

## Protocol shared by every DeepFashion notebook

| | setting |
|---|---|
| base | the U-Net of the Modality-1 base notebook, 64 base channels, verbatim; its weights from `cfg_fashion.ipynb` (`checkpoints/diffusion_fashion/best.pt` or `FASHION_BASE_CKPT`), or trained here with the same recipe if absent |
| base recipe | AdamW 2e-4, constant learning rate, mixed precision, gradient clip 1, 10% label dropout, batch 64, 50 epochs, best validation checkpoint |
| schedule | their cosine schedule, K = 1000, betas clamped to [1e-4, 0.9999], epsilon prediction |
| sampler | ancestral DDPM sampling (the DDIM step with η = 1, fresh noise every step), 50 steps, clean estimate clamped to [-1, 1]; `eta = 0` switches every notebook to deterministic DDIM |
| condition text | the image's caption from `captions.csv`, frozen CLIP ViT-B/32 |
| evaluation | 2048 class-balanced validation images; one generated image per validation caption or class; scorer = fine-tuned ResNet-18; FID, top-1 / top-5, precision / recall |
| seeds | `seed` for initialisation, training and sampling noise; `split_seed` for the validation subset |
| quick run | `%env CFG_QUICK=1` in a cell above the config, in all four notebooks: a class-balanced 10k of the 209k training images, 8 base and 3 adapter epochs, 256 evaluation images. Shows that everything works in well under an hour; its numbers are not results |

## How this notebook implements TFG

**Notation.** $x_k$ is the noisy image at diffusion step $k$ ($K = 1000$; the network receives
$t = k/K$), $k'$ the next step of the $N = 50$-step DDIM schedule, $\bar\alpha_k$ the cosine
schedule, $\varepsilon_\theta$ the frozen U-Net, $\varnothing$ the null label. $y$ is the target
garment category and $p_r(y \mid x)$ a frozen classifier on clean images: an ImageNet-pretrained
ResNet-34 fine-tuned on the real DeepFashion training images, then frozen.

**The paper's idea.** Steer an *unconditional* diffusion model with any differentiable objective
on clean data, without training anything: at every step, estimate the clean image, score it,
and move along the gradient. TFG shows that earlier training-free methods are special cases of
two terms: a *variance* term, the gradient with respect to the noisy state taken through the
denoiser, and a *mean* term, gradient steps taken on the clean estimate itself. Both act on a
smoothed objective, and both have a schedule over the trajectory. Per step $k \to k'$
(Algorithm 1):

$$\hat x_0 = \operatorname{clip}\!\Big(\frac{x_k - \sqrt{1-\bar\alpha_k}\;\varepsilon_\theta(x_k, k, \varnothing)}{\sqrt{\bar\alpha_k}}\Big),
\qquad
\log \tilde f(x) = \log \frac{1}{n_\varepsilon}\sum_{i=1}^{n_\varepsilon} \exp f\big(x + \sigma^{\mathrm s}_k\,\delta_i\big),\quad f(x) = \log p_r(y \mid x)$$

$$\Delta_k = \rho_k\, \nabla_{x_k} \log \tilde f\big(\hat x_0(x_k)\big)
\qquad\text{(variance guidance; the gradient goes through } \varepsilon_\theta)$$

$$\hat x_0^{(j+1)} = \hat x_0^{(j)} + \mu_k\, \nabla \log \tilde f\big(\hat x_0^{(j)}\big),\ \ j = 0,\dots,N_{\text{iter}}-1,
\qquad \Delta_0 = \hat x_0^{(N_{\text{iter}})} - \hat x_0
\qquad\text{(mean guidance)}$$

$$x_{k'} = \underbrace{\sqrt{\bar\alpha_{k'}}\,\hat x_0 + \sqrt{1-\bar\alpha_{k'}}\,\hat\varepsilon}_{\text{plain DDIM step, } \hat\varepsilon \text{ recomputed from the clipped } \hat x_0}
\;+\; \frac{\Delta_k}{\sqrt{\bar\alpha_k / \bar\alpha_{k'}}} \;+\; \sqrt{\bar\alpha_{k'}}\;\Delta_0$$

Schedules are theirs: $\rho_k = \rho\, a_k / \bar a$ and $\mu_k = \mu\, a_k / \bar a$ with
$a_k = \bar\alpha_k / \bar\alpha_{k'}$ and $\bar a$ its mean over the trajectory, so the mean
strength equals the $\rho$ and $\mu$ you set; $\sigma^{\mathrm s}_k = \sigma^{\mathrm s}\sqrt{1-\bar\alpha_k}$.
(The paper writes the smoothing scale $\sigma_t$; it is $\sigma^{\mathrm s}$ here so that $\sigma$
is not overloaded.)

**How it is used here.** The unconditional denoiser is the shared class-conditional DeepFashion U-Net run
with the null label. The objective is $\log p_r(y \mid \hat x_0)$ from the fine-tuned ResNet-34; the
classifier wrapper takes images in $[-1, 1]$, upsamples to 128×128 and applies ImageNet
normalisation, so `to_clf` is the identity here. The hyperparameters are
their CIFAR-10 label-guidance script, unchanged. The DDIM step inside the guided sampler is the
same step the rest of this repository uses, and the first cell after setup asserts that at
$\rho = \mu = 0$ and $\eta = 0$ the guided sampler reproduces plain DDIM exactly. Nothing is trained. Each
guided step costs one denoiser forward, one backward, and $N_{\text{iter}} + 1$ classifier
evaluations; the notebook reports the counts rather than a single NFE.

| In the paper / official code | In this notebook |
|---|---|
| Alg. 1, clean estimate $\hat x_0$, clamped | `TFG._predict_x0` in `tfg.py` |
| Alg. 1, variance guidance $\Delta_k$, gradient through the denoiser | the `rho` block of `TFG.sample` |
| Alg. 1, mean guidance, $N_{\text{iter}}$ steps on the clean estimate | the `iter_steps` loop of `TFG.sample` |
| Alg. 1, the update $x_{k'}$; `_predict_x_prev_from_zero` recomputes $\hat\varepsilon$ from the clipped $\hat x_0$ | `TFG.sample`; `ddim_sample` here uses the same convention, and *Set up TFG* asserts the two agree at $\rho = \mu = 0$ |
| $\log\tilde f$, `tilde_get_guidance` | `TFG._log_f_tilde` |
| schedules `get_rho`, `get_mu`, `get_std` | `TFG._strength`, `TFG._std` |
| `ImageLabelGuidance`: the objective is the classifier's $\log p(y \mid \hat x_0)$ | `make_log_p` on the fine-tuned ResNet-34 (`guide_clf`) |
| `scripts/cifar10_label.sh`: $\rho = 1$, $\mu = 0.25$, $\sigma^{\mathrm s} = 10^{-3}$, $n_\varepsilon = 1$, $N_{\text{iter}} = 4$, one recurrence, 50 steps | the `Cfg` defaults |
| an unconditional diffusion model | `eps_uncond`: our class-conditional model run with the null label |
| not in TFG: a second classifier for scoring, the wrong-target row, paired generators, the strength sweep | *Sample*, *Condition accuracy*, *Guidance-strength sweep*: our evaluation protocol |

**What the paper did that this notebook does not.** Their sampler is stochastic DDIM
with $\eta = 1$, the ancestral DDPM sampler, and so is ours here (50 steps; `eta = 0` gives
deterministic DDIM). Their diffusion model is an unconditional pretrained CIFAR-10
DDPM; ours is the DeepFashion class-conditional U-Net standing in as unconditional through the
null label. Their classifier is pretrained; ours is fine-tuned here, once, on real images only.
They searched hyperparameters per task; we take their CIFAR-10 values as they are, so these
settings are not tuned for DeepFashion, and the strength sweep below is the first look at that. Six deviations are listed in the header of
`src/guidance/baselines/tfg.py`.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("datasets",):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    # Inside a checkout (Jupyter on a server, cwd somewhere under the repo): walk up to its src/.
    from pathlib import Path as _P
    _src = next((p / "src" for p in [_P.cwd(), *_P.cwd().parents] if (p / "src" / "guidance").is_dir()), None)
    if _src is None:                                   # Colab: clone the repo next to the notebook
        if not os.path.exists("discrete-generative-models-proj-1"):
            subprocess.run(["git", "clone", "-q",
                            "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                           check=True)
        else:                                          # cloned earlier next to this notebook: update it
            subprocess.run(["git", "-C", "discrete-generative-models-proj-1", "pull", "-q"], check=False)
        _src = "discrete-generative-models-proj-1/src"
    sys.path.insert(0, str(_src))
    print("guidance package from:", _src)

import torch
from guidance.baselines.tfg import TFG, TFGConfig

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 42                 # training / sampling randomness, as cfg_fashion
    split_seed: int = 42           # validation subset; fixed so multi-seed runs share one subset
    batch_size: int = 64
    base_epochs: int = 50
    lr_base: float = 2e-4
    lr_schedule: str = "constant"  # the DeepFashion base recipe
    amp: bool = True               # fp16 autocast on CUDA, as the DeepFashion base recipe
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 50         # sampling steps; TFG's CIFAR-10 script also uses 50
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0: a class-balanced seeded subset of the training images (quick or smoke runs)
    val_limit: int = 2048          # class-balanced validation subset, as the base notebook
    base_channels: int = 64        # width of the DeepFashion U-Net, as the base notebook
    n_eval: int = 2048             # generated images per arm for FID, accuracy, precision / recall
    clf_epochs: int = 10           # fine-tuning epochs of the scorer, the base notebooks' recipe
    quick: bool = False            # showcase budget (CFG_QUICK=1): 10k images, 8 base / 3 adapter epochs, 256 eval images
    n_per_class: int = 2           # samples per active class in the paired four-row comparison and the sweep
    # TFG -- defaults are the official CIFAR-10 label-guidance script
    rho: float = 1.0
    mu: float = 0.25
    sigma: float = 0.001
    eps_bsz: int = 1
    iter_steps: int = 4
    recur_steps: int = 1
    eta: float = 1.0               # ancestral DDPM sampling, as every DeepFashion notebook and TFG's own script; 0 = deterministic DDIM
    guide_clf: str = "resnet34"    # steers the sampler; fine-tuned here on real training images
    eval_clf: str = "resnet18"     # scores the result, as every DeepFashion notebook; must differ from guide_clf

cfg = Cfg()
if os.environ.get("CFG_QUICK", "").lower() in ("1", "true", "yes"):
    cfg.quick = True
if cfg.quick:
    # The showcase budget, the same in every DeepFashion notebook so they share one class-balanced
    # 10k subset and one base. Explicit CFG_* variables below still override it. Not a result.
    for _k, _v in dict(limit=10_000, base_epochs=8, lora_epochs=3, xattn_epochs=3,
                       n_eval=256, clf_epochs=3, n_per_class=1).items():
        if hasattr(cfg, _k):
            setattr(cfg, _k, _v)
    print("QUICK showcase budget:", {k: getattr(cfg, k) for k in ("limit", "base_epochs", "n_eval", "clf_epochs")})

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
assert cfg.guide_clf != cfg.eval_clf, "guide and eval classifiers must be different networks"
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = os.environ.get("FASHION_CKPT_DIR", "checkpoints")   # shared base and classifier checkpoints

def clf_search(arch):
    '''Where a fine-tuned classifier may already be: this folder's checkpoints/, the repo's (two
    levels up from notebooks/fashion/), data/, or FASHION_CLF_DIR. The flow-matching base notebook
    saves its scorer as fashion_resnet18_classifier.pt; drop it in and no fine-tuning runs.'''
    dirs = [os.environ.get("FASHION_CLF_DIR"), CKPT_DIR, "../checkpoints", "../../checkpoints", "data", "../../data"]
    return [f"{d}/fashion_{arch}_classifier.pt" for d in dirs if d]
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data source

DeepFashion, Category and Attribute Prediction benchmark (Liu et al., CVPR 2016), with one
caption per image in `captions.csv`. Unzip the benchmark archive under `data/` in the repo (or
next to the notebook), unzip the image archive inside it so that it holds `img/`, `Anno_coarse/`
and `Eval/`, and put `captions.csv` in that folder or one folder above it. The loader looks up to
two levels under `data/`; `FASHION_ROOT` and `FASHION_CAPTIONS` point it somewhere else.

## Data

The loader is the one from the Modality-1 base notebooks, imported from
`src/guidance/data/fashion.py`: crop to the garment box, pad to a white square, 64×64, random
horizontal flips for training, pixels in `[-1, 1]`, the official train / validation partition,
and a class-balanced validation subset of 2048 images. 46 of the 50 coarse categories have
training images.

Each loader yields `(x, idx)`: the image and its position in its split. Two lookups turn the
position into a condition: `LAB_TR[idx]` / `LAB_VA[idx]` is the **class label** the backbone was
trained on (50 categories, null token 50), and `CAP_TR[idx]` / `CAP_VA[idx]` is the image's
**caption**, e.g. "This is a chic sleeveless pleated blouse with a v-neckline." The caption is
the text the text-conditioned arms read: it names the category and attributes (sleeve, neckline,
pattern, fabric) that the 50-way label does not carry. This is what the text can add over the label.

In [ ]:
from guidance.data.fashion import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, limit=cfg.limit, val_limit=cfg.val_limit)

LAB_TR, LAB_VA = meta["labels_train"].to(DEV), meta["labels_val"].to(DEV)      # position -> class label
CAP_TR, CAP_VA = meta["captions_train"], meta["captions_val"]                  # position -> caption
ACTIVE = torch.tensor(meta["active_class_ids"], device=DEV)                    # classes with training images
N_VAL = meta["n_val"]

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"[{float(xb.min()):.2f}, {float(xb.max()):.2f}]", "| idx", tuple(cb.shape))
print(f"train {meta['n_train']:,} | val {meta['n_val']:,} | classes {meta['n_classes']} "
      f"({len(meta['active_class_ids'])} with training images) | root {meta['root']}")
i0 = int(cb[0])
print(f"first item: class '{meta['class_names'][int(LAB_TR[i0])]}' | caption: {CAP_TR[i0]}")

## Two fine-tuned classifiers: one guides, one scores

No pretrained DeepFashion classifier exists, so two ImageNet-pretrained ResNets are fine-tuned on
the real training images with the recipe of the flow-matching base notebook (AdamW 1e-3, cosine
annealing, 10 epochs, mixed precision), once, and cached under `checkpoints/`. **ResNet-34
guides** the sampler; **ResNet-18 scores** the result, and is the same scorer every DeepFashion
notebook uses, so accuracies compare across arms. Neither ever sees a generated image during
training.

The wrapper takes images in `[-1, 1]`, the diffusion model's range, upsamples to 128×128 and
applies ImageNet normalisation, so no conversion is needed here (`to_clf` is the identity). The
clamp inside the wrapper is part of the guidance objective: out-of-range pixels are not images,
and there the objective's gradient is zero; saturation is reported below. The cell prints both
classifiers' accuracy on real validation images: the guide's says the objective is meaningful,
the scorer's is the ceiling for every accuracy below.

In [ ]:
from guidance.eval.image_metrics import fit_classifier, real_accuracy, top_k_hits, precision_recall_features, to_uint8

def to_clf(x):
    '''Identity: the classifier wrapper converts from [-1, 1] itself (clamp, upsample, normalise).'''
    return x

def fit(arch):
    clf = fit_classifier(arch, meta["n_classes"], train_loader, val_loader, LAB_TR, LAB_VA, DEV,
                         f"{CKPT_DIR}/fashion_{arch}_classifier.pt", epochs=cfg.clf_epochs, seed=cfg.seed,
                         search=clf_search(arch))
    for p in clf.parameters():
        p.requires_grad_(False)
    return clf.eval()

guide_clf = fit(cfg.guide_clf)
eval_clf = fit(cfg.eval_clf)

REAL_ACC = {}
for name, clf in [(cfg.guide_clf, guide_clf), (cfg.eval_clf, eval_clf)]:
    REAL_ACC[name] = real_accuracy(clf, val_loader, LAB_VA, DEV)
    print(f"{name:>9}: top-1 {REAL_ACC[name][0]:.3f}, top-5 {REAL_ACC[name][1]:.3f} on {N_VAL} real validation images "
          f"(chance {1 / len(ACTIVE):.3f})")
SCORER_REAL_TOP1, SCORER_REAL_TOP5 = REAL_ACC[cfg.eval_clf]

@torch.no_grad()
def clf_features(x):
    return torch.cat([eval_clf.features(x[i:i + 256]) for i in range(0, len(x), 256)])

@torch.no_grad()
def hits(x, y):
    '''top-1 / top-5 hit tensors under the scorer, in batches.'''
    h1, h5 = [], []
    for i in range(0, len(x), 256):
        a, b = top_k_hits(eval_clf(x[i:i + 256]), y[i:i + 256])
        h1.append(a); h5.append(b)
    return torch.cat(h1), torch.cat(h5)

REAL_FEATS = torch.cat([clf_features(x.to(DEV)) for x, _ in val_loader])

## Backbone

In [ ]:
# ---- The model of notebooks/fashion/CIS6720_Proj1_DiffusionCode_Modality_1.ipynb, a teammate's
# class-conditional base for DeepFashion, kept verbatim. The class label is added to the time
# embedding and enters every residual block; bottleneck self-attention at 16x16.
base_channels = cfg.base_channels
num_classes = meta["n_classes"]


def timestep_embedding(k, width):
    '''Sinusoidal timestep features, before a learned projection.'''
    half = width // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half, device=k.device)
                            / max(half - 1, 1))
    angles = k.float()[:, None] * frequencies[None]
    result = torch.cat([angles.sin(), angles.cos()], dim=-1)
    return F.pad(result, (0, width - result.shape[-1]))


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, embedding_width):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.embedding = nn.Linear(embedding_width, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = (nn.Conv2d(in_channels, out_channels, 1)
                     if in_channels != out_channels else nn.Identity())

    def forward(self, x, embedding):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.embedding(F.silu(embedding))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Attention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        q = q.flatten(2).transpose(1, 2)
        k = k.flatten(2).transpose(1, 2)
        v = v.flatten(2).transpose(1, 2)
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(1, 2).reshape(batch, channels, height, width)
        return x + self.out(h)


class DiffusionUNet(nn.Module):
    def __init__(self, classes=num_classes, base=base_channels):
        super().__init__()
        self.base = base
        width = base * 4
        self.time_mlp = nn.Sequential(nn.Linear(base, width), nn.SiLU(), nn.Linear(width, width))
        self.class_embedding = nn.Embedding(classes + 1, width)
        self.input = nn.Conv2d(3, base, 3, padding=1)
        self.down1 = ResBlock(base, base, width)
        self.stride1 = nn.Conv2d(base, base, 3, stride=2, padding=1)
        self.down2 = ResBlock(base, base * 2, width)
        self.stride2 = nn.Conv2d(base * 2, base * 2, 3, stride=2, padding=1)
        self.middle1 = ResBlock(base * 2, base * 2, width)
        self.attention = Attention(base * 2)  # Bottleneck self-attention at 16×16.
        self.middle2 = ResBlock(base * 2, base * 2, width)
        self.up2 = ResBlock(base * 4, base * 2, width)
        self.up1 = ResBlock(base * 3, base, width)
        self.output_norm = nn.GroupNorm(8, base)
        self.output = nn.Conv2d(base, 3, 3, padding=1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, xk, k, class_labels):
        embedding = (self.time_mlp(timestep_embedding(k, self.base))
                     + self.class_embedding(class_labels))
        x = self.input(xk)
        skip1 = self.down1(x, embedding)              # [B, base, 64, 64]
        skip2 = self.down2(self.stride1(skip1), embedding)  # [B, 2*base, 32, 32]
        x = self.middle2(self.attention(self.middle1(self.stride2(skip2), embedding)), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up2(torch.cat([x, skip2], dim=1), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up1(torch.cat([x, skip1], dim=1), embedding)
        return self.output(F.silu(self.output_norm(x)))  # Predict RGB noise.


# ---- Our wrapper. The notebooks pass t in [0, 1] and an optional class id; None is the null
# token, their `null_class = num_classes`. t is scaled back to the integer step their time
# embedding was written for.
class FashionUNet(DiffusionUNet):
    def __init__(self):
        super().__init__(classes=meta["n_classes"], base=cfg.base_channels)
        self.null = meta["n_classes"]

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        return super().forward(x, t * cfg.K, cond_ids)


def make_backbone():
    return FashionUNet()

# Only the trunk is adapted by TC-LoRA: the input convolution, every residual block's two 3x3
# convolutions and 1x1 skip, and the attention's qkv / out projections. The time MLP, the class
# embedding, each block's embedding projection (the conditioning path) and the output layer are
# not; the strided downsampling convolutions cannot be (deviation 1).
SKIP_NAMES = ("time_mlp", "class_embedding", ".embedding", "output")

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`
net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion: cosine schedule, epsilon prediction, ancestral sampler

The sampler is the DDIM step (Song et al. 2020, eq. 12) with `eta = 1`, which is ancestral DDPM
sampling: fresh noise is added at every step, drawn from a seeded generator so paired rows stay
paired. `eta = 0` gives deterministic DDIM. Every DeepFashion notebook uses `eta = 1`, 50 steps and
the clean estimate clamped to [-1, 1].

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

def cosine_alpha_bar_diffusers(K, max_beta=0.999, device="cpu"):
    '''diffusers "squaredcos_cap_v2": the same cosine curve turned into betas clipped at
    max_beta, then a cumulative product. The schedule the cell base (cfg_cells) was trained with.'''
    steps = torch.arange(K + 1, dtype=torch.float64, device=device) / K
    f = torch.cos((steps + 0.008) / 1.008 * math.pi / 2) ** 2
    betas = (1 - f[1:] / f[:-1]).clamp(max=max_beta)
    return torch.cumprod(1 - betas, 0).float()

def cosine_alpha_bar_fashion(K, device="cpu"):
    '''The schedule of the DeepFashion base notebook, verbatim: the cosine curve turned into betas
    clamped to [1e-4, 0.9999], then a cumulative product. Sharing the base checkpoint requires
    sampling with the schedule it was trained under.'''
    step_fraction = torch.linspace(0, K, K + 1, device=device, dtype=torch.float64) / K
    cosine_curve = torch.cos((step_fraction + 0.008) / 1.008 * math.pi / 2).square()
    cosine_curve = cosine_curve / cosine_curve[0]
    betas = (1 - cosine_curve[1:] / cosine_curve[:-1]).clamp(0.0001, 0.9999).float()
    return torch.cumprod(1 - betas, dim=0)

ABAR = cosine_alpha_bar_fashion(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]
CLIP_X0 = 1.0


def add_noise(x0, gen=None):
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, x_init=None, eta=None, generator=None):
    '''The DDIM step, TFG's convention: clip x0, then RECOMPUTE eps from the clipped x0 before
    stepping. eta = 1 (the default here) adds the ancestral noise; eta = 0 is deterministic DDIM. Keeping the original eps is a different update whenever
    the clamp engages; the two can differ by O(1) on real images. The TFG sampler in
    src/ reduces to exactly this at rho = mu = 0, and the cell below asserts it.'''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    eta = cfg.eta if eta is None else eta
    x = torch.randn(n, *shape, device=DEV, generator=generator) if x_init is None else x_init.clone()
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        if clip_x0:
            x0 = x0.clamp(-clip_x0, clip_x0)
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()          # recompute from (the clipped) x0, as TFG does
        # eta = 0 is deterministic DDIM; eta = 1 is ancestral DDPM (DDIM paper eq. 16), with fresh
        # noise at every step drawn from `generator` so paired rows stay paired.
        sigma = eta * ((1 - ab_prev) / (1 - ab) * (1 - ab / ab_prev)).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        if eta > 0:
            x = x + sigma * torch.randn(x.shape, device=DEV, generator=generator)
    return x

## Base model: load the shared DeepFashion base, or train it

Identical to the other arms: `checkpoints/diffusion_fashion/best.pt` from `cfg_fashion.ipynb`
(or `base.pt` from an earlier run) is loaded after a check of split, schedule and backbone, so
every DeepFashion notebook uses the same frozen weights; if neither is present the same
recipe trains here.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    # cfg.lr_schedule: "cosine" anneals to zero over the run (images); "constant" holds lr, the
    # cell base's recipe. cfg.amp: fp16 autocast with a grad scaler on CUDA, also the cell base's
    # recipe; the loss itself is computed in float32 either way.
    opt = torch.optim.AdamW(params, lr=lr)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1)) if cfg.lr_schedule == "cosine"
             else torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0))
    use_amp = bool(cfg.amp) and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()
        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    # y is the position in the training split; the backbone is trained on the class label with
    # 10% label dropout, the recipe of the DeepFashion base notebook.
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(LAB_TR[y])), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, LAB_VA[y]), eps)


CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine_fashion",
             "backbone_params": sum(p.numel() for p in net.parameters())}

if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
elif cfg.reuse_base and (_cfg_ckpt := next(
        (p for p in [os.environ.get("FASHION_BASE_CKPT"), f"{CKPT_DIR}/diffusion_fashion/best.pt",
                     "../checkpoints/diffusion_fashion/best.pt", "../../checkpoints/diffusion_fashion/best.pt"]
         if p and os.path.exists(p)), None)):
    # The base trained by cfg_fashion.ipynb: same model class, so the weights load directly. Every
    # DeepFashion notebook then uses the same base. Its validation loss was computed with
    # that notebook's noise draws, so it is reported as it was recorded, not recomputed.
    ck = torch.load(_cfg_ckpt, map_location=DEV, weights_only=False)
    assert ck["meta"]["n_params"] == sum(p.numel() for p in net.parameters()) and \
        ck["meta"]["num_train_timesteps"] == cfg.K, f"{_cfg_ckpt} does not match this backbone"
    net.load_state_dict(ck["model"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val_loss"], "best": True}]
    torch.save({"state": net.state_dict(), "epoch": ck["epoch"], "val": ck["val_loss"], "meta": CKPT_META,
                "source": _cfg_ckpt}, "base.pt")
    print(f"loaded the cfg_fashion base from {_cfg_ckpt} (epoch {ck['epoch']}, val {ck['val_loss']:.4f}); "
          f"saved as base.pt; skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    ck = torch.load("base.pt", map_location=DEV); ck["meta"] = CKPT_META; torch.save(ck, "base.pt")

# Whichever branch ran, sample from the BEST checkpoint on disk, not the last epoch left in
# `net`. Without this a fresh run and a reused-checkpoint run would use different weights
# while the manifest reported the same best validation loss for both.
ck = torch.load("base.pt", map_location=DEV)
net.load_state_dict(ck["state"])
base_frozen = net.eval()
for p in base_frozen.parameters():
    p.requires_grad_(False)
print(f"sampling from base.pt epoch {ck['epoch']} (val {ck['val']:.4f})")

## Set up TFG

TFG expects an **unconditional** denoiser. Ours is class-conditional with a null label, so
`eps_uncond` runs it with the null label throughout: the label path is off, and the
classifier gradient is the only way the class enters.

The objective is `log p(y | x̂₀)` from the frozen guide classifier (the fine-tuned ResNet-34),
evaluated on the predicted clean image. That is what
their `ImageLabelGuidance` returns.

First check: at `rho = mu = 0` the TFG sampler must reproduce plain DDIM to numerical
precision. If it does not, the guided sampler is wrong and nothing below can be used.

In [ ]:
def eps_uncond(x, t):
    # TFG assumes an unconditional denoiser. Ours is class-conditional with a null label, so the
    # null label stands in for "unconditional" and the classifier gradient is the only way in.
    return base_frozen(x, t, torch.full((x.shape[0],), NULL_ID, device=x.device, dtype=torch.long))


def make_log_p(y):
    '''log p(y | x0) under the guide classifier, for a fixed batch of targets y.
    This is the ImageLabelGuidance objective: the quantity Algorithm 1 climbs at every step.'''
    def log_p(x0):
        logits = guide_clf(to_clf(x0))
        yy = y.repeat(x0.shape[0] // y.shape[0])          # smoothing may stack eps_bsz copies
        return F.log_softmax(logits, dim=-1).gather(1, yy[:, None]).squeeze(1)
    return log_p


tfg_cfg = TFGConfig(rho=cfg.rho, mu=cfg.mu, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                    iter_steps=cfg.iter_steps, recur_steps=cfg.recur_steps,
                    clip_x0=CLIP_X0, eta=cfg.eta)

# sanity: guidance off == plain DDIM, same starting noise
g = torch.Generator(device=DEV).manual_seed(0)
x0_probe = torch.randn(8, *meta["shape"], device=DEV, generator=g)
y_probe = ACTIVE[:8]
off = TFG(eps_uncond, make_log_p(y_probe), ABAR, TFGConfig(rho=0, mu=0, iter_steps=0, clip_x0=CLIP_X0))
a = off.sample(8, meta["shape"], 10, x_init=x0_probe)
b = ddim_sample(eps_uncond, 8, meta["shape"], 10, x_init=x0_probe, eta=0.0)   # deterministic on both sides
assert torch.allclose(a, b, atol=1e-4), (a - b).abs().max().item()
assert off.stats.objective_evals == 0 and off.stats.denoiser_backward == 0
print("rho = mu = 0 reproduces plain DDIM exactly")
print("TFG config:", asdict(tfg_cfg))

## Sample: four rows from the same starting noise

| row | what | label path | guidance |
|---|---|---|---|
| **uncond** | base, null label | off | none |
| **label** | base, true label | on | none; the reference for what the label is worth |
| **TFG** | base, null label | off | classifier gradient toward `y` |
| **TFG, wrong y** | base, null label | off | classifier gradient toward `y + 1` |

The last row is the check: if TFG works, its samples should score as `y + 1`, not
`y`. Guidance that ignores its target would leave rows 3 and 4 identical.

Every row is timed. TFG costs a denoiser backward pass and several classifier passes per
step, so its cost is reported as a breakdown rather than one number.

**Pairing.** The same starting noise is not enough: TFG draws smoothing noise inside the
loop (`sigma > 0`), and with recurrence it would draw re-noising too. Every TFG call in a
comparison gets its own generator seeded identically, so the *only* thing that differs
between rows is the target. `n_per_class` per active class (46 of the 50 have training images), so every class the base
knows is in the comparison.

In [ ]:
y_eval = ACTIVE.repeat(cfg.n_per_class)                     # every active class, n_per_class times
y_wrong = ACTIVE.roll(-1).repeat(cfg.n_per_class)           # the next active class
N = len(y_eval)
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

def paired_gen():
    '''Fresh generator, same seed: identical internal randomness for every matched call.'''
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 1234)

def timed(fn):
    if DEV.type == "cuda": torch.cuda.synchronize()
    t0 = time.time(); out = fn()
    if DEV.type == "cuda": torch.cuda.synchronize()
    return out, (time.time() - t0) / N

rows = {}
rows["uncond"], t_uncond = timed(lambda: ddim_sample(eps_uncond, N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
rows["label"], t_label = timed(lambda: ddim_sample(lambda x, t: base_frozen(x, t, y_eval), N, meta["shape"],
                                                   cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg = TFG(eps_uncond, make_log_p(y_eval), ABAR, tfg_cfg)    # Algorithm 1, their CIFAR-10 settings
rows["tfg"], t_tfg = timed(lambda: tfg.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg_stats = tfg.stats
tfg_w = TFG(eps_uncond, make_log_p(y_wrong), ABAR, tfg_cfg)
rows["tfg_wrong"], _ = timed(lambda: tfg_w.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))

# How often the guided clean estimate hit the clamp -- the objective's gradient is zero there.
SATURATION = {k: (v.abs() >= 0.999).float().mean().item() for k, v in rows.items()}

print(f"{'row':>10} {'s / sample':>11}   denoiser fwd / bwd   objective calls   saturated px")
print(f"{'uncond':>10} {t_uncond:>11.3f}   {cfg.sample_steps:>4} / {0:<4}          0          {SATURATION['uncond']:6.1%}")
print(f"{'label':>10} {t_label:>11.3f}   {cfg.sample_steps:>4} / {0:<4}          0          {SATURATION['label']:6.1%}")
print(f"{'tfg':>10} {t_tfg:>11.3f}   {tfg_stats.denoiser_evals:>4} / {tfg_stats.denoiser_backward:<4}          "
      f"{tfg_stats.objective_evals}        {SATURATION['tfg']:6.1%}")

## Condition accuracy, scored by the other classifier

`eval_clf` was not used by the sampler. Its accuracy on each row is the condition metric.
For the wrong-target row the number reported is accuracy against `y + 1`: high means
guidance followed its target.

In [ ]:
@torch.no_grad()
def acc_of(x, y):
    return (eval_clf(to_clf(x)).argmax(1) == y).float().mean().item()

ACC = {
    "uncond": acc_of(rows["uncond"], y_eval),
    "label": acc_of(rows["label"], y_eval),
    "tfg": acc_of(rows["tfg"], y_eval),
    "tfg_wrong_vs_target": acc_of(rows["tfg_wrong"], y_wrong),
    "tfg_wrong_vs_original": acc_of(rows["tfg_wrong"], y_eval),
}
for k, v in ACC.items():
    print(f"{k:>24}: {v:6.1%}")
print()
print(f"chance = {1 / len(ACTIVE):.1%}; the scorer's top-1 on real validation images is {SCORER_REAL_TOP1:.1%}.")
print("If guidance follows its target, 'tfg' sits well above 'uncond' and 'tfg_wrong_vs_target' is "
      "comparably high. 'tfg_wrong_vs_original' is informative but not a pass/fail: it can sit "
      "below chance, or above it through systematic class confusion, without guidance being wrong.")
if ACC["tfg"] < ACC["uncond"] + 0.05:
    print("little steering at these settings: try larger rho/mu or more iter_steps, and re-check to_clf.")
print()
print("Accuracy alone rewards unrealistic images even with a separate evaluator. Read it with FID and precision / recall below.")

## Samples

In [ ]:
import matplotlib.pyplot as plt

def short(i):
    return meta["class_names"][int(i)]

show = min(8, N)
fig, axes = plt.subplots(4, show, figsize=(2 * show, 8.6))
for r, (name, s, ys) in enumerate([("uncond", rows["uncond"], y_eval), ("label", rows["label"], y_eval),
                                   ("TFG", rows["tfg"], y_eval), ("TFG, wrong y", rows["tfg_wrong"], y_wrong)]):
    for j in range(show):
        axes[r, j].imshow(((s[j].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
        if r in (2, 3):
            axes[r, j].set_title(short(ys[j]), fontsize=8)
    axes[r, 0].set_ylabel(name, fontsize=10)
plt.tight_layout(); plt.show()

## Guidance-strength sweep

`rho` and `mu` set the guidance strength. A short sweep at fixed `mu/rho` ratio shows the
accuracy-vs-time trade-off. This is the analogue of a classifier-free-guidance strength
sweep; a learned guidance controller is compared against the best point on this curve, not
against the default setting.

This is a first pass, not TFG's own hyperparameter search, and it is run on validation
targets. Pick a setting here, then report it with fresh sampling seeds; the maximum of the sweep
itself is not the result.

In [ ]:
SWEEP = [0.0, 0.5, 1.0, 2.0, 4.0]
sweep_rows = []
for scale in SWEEP:
    c = TFGConfig(rho=cfg.rho * scale, mu=cfg.mu * scale, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                  iter_steps=cfg.iter_steps if scale > 0 else 0, recur_steps=cfg.recur_steps,
                  clip_x0=CLIP_X0, eta=cfg.eta)
    s, dt = timed(lambda: TFG(eps_uncond, make_log_p(y_eval), ABAR, c).sample(
        N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
    sweep_rows.append({"scale": scale, "rho": c.rho, "mu": c.mu, "acc": acc_of(s, y_eval), "s_per_sample": dt})
    print(f"scale {scale:>4}  rho {c.rho:>5.2f}  mu {c.mu:>5.3f}  acc {sweep_rows[-1]['acc']:6.1%}  {dt:.3f}s/sample")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot([r["scale"] for r in sweep_rows], [r["acc"] for r in sweep_rows], marker="o")
ax.axhline(ACC["label"], ls="--", c="grey", lw=1, label=f"label-conditioned base ({ACC['label']:.1%})")
ax.axhline(1 / len(ACTIVE), ls=":", c="grey", lw=1, label="chance")
ax.set_xlabel("guidance scale (x default rho, mu)"); ax.set_ylabel(f"accuracy ({cfg.eval_clf})")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Evaluation on the validation set: accuracy, FID, precision and recall

The protocol of the DeepFashion base notebooks, applied per arm. One image is generated for each
of the `n_eval` validation images, targeting **that image's class**: the base with the null
label (unconditional), the base with the true label (the reference for what the label is
worth), and TFG steering the null-label base toward the class with the guide classifier. Then:

- **Condition accuracy**: the scorer's top-1 / top-5 against the target class. The scorer is a
  different network from the guide, so guidance is not judged by its own objective.
- **FID** (torchmetrics, Inception pool features) against the same validation images. With
  2048 on each side FID is biased upward; the count is recorded next to the number.
- **Precision and recall** in the scorer's feature space (k = 3 nearest neighbours).

Same starting noise and same per-step generator for every arm, so the rows are paired. TFG's
cost per sample is reported next to its accuracy. `n_eval` at 2048 is the full protocol.

In [ ]:
def _sample(eps_fn, idx, x0, gen):
    return ddim_sample(eps_fn, len(idx), meta["shape"], cfg.sample_steps, x_init=x0, generator=gen)

ARMS = [
    ("uncond", lambda idx, x0, gen: _sample(eps_uncond, idx, x0, gen),                                   lambda idx: LAB_VA[idx]),
    ("label",  lambda idx, x0, gen: _sample(lambda x, t: base_frozen(x, t, LAB_VA[idx]), idx, x0, gen),  lambda idx: LAB_VA[idx]),
    ("tfg",    lambda idx, x0, gen: TFG(eps_uncond, make_log_p(LAB_VA[idx]), ABAR, tfg_cfg).sample(
                   len(idx), meta["shape"], cfg.sample_steps, x_init=x0, generator=gen),               lambda idx: LAB_VA[idx]),
]

EVAL_IDX = torch.arange(min(cfg.n_eval, N_VAL), device=DEV)
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 3)
X0_EVAL = torch.randn(len(EVAL_IDX), *meta["shape"], device=DEV, generator=g)

def eval_gen():
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 4)

def generate(sample_fn):
    '''All n_eval images from the same starting noise, in batches, one generator stream.'''
    out, gen = [], eval_gen()
    for i in range(0, len(EVAL_IDX), 128):
        out.append(sample_fn(EVAL_IDX[i:i + 128], X0_EVAL[i:i + 128], gen))
    return torch.cat(out)

def fid_of(x):
    import importlib.util, subprocess, sys
    for pkg, mod in [("torchmetrics", "torchmetrics"), ("torch-fidelity", "torch_fidelity")]:
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEV)
    fid.set_dtype(torch.float64)
    for xr, _ in val_loader:
        fid.update(to_uint8(xr.to(DEV)), real=True)
    for i in range(0, len(x), 256):
        fid.update(to_uint8(x[i:i + 256]), real=False)
    return float(fid.compute())

METRICS = {"n_eval": int(len(EVAL_IDX)), "n_real": N_VAL, "scorer": "fine-tuned ResNet-18 (never used by the sampler)",
           "scorer_real_val_top1": SCORER_REAL_TOP1, "scorer_real_val_top5": SCORER_REAL_TOP5,
           "precision_recall_k": 3, "arms": {}}
EVAL_SAMPLES = {}
print(f"{'arm':>14} {'top-1':>7} {'top-5':>7} {'FID':>8} {'prec':>6} {'rec':>6}   {'s/sample':>8}   (n = {len(EVAL_IDX)}; "
      f"real top-1 {SCORER_REAL_TOP1:.3f}; chance {1 / len(ACTIVE):.3f})")
for name, sample_fn, target_fn in ARMS:
    if DEV.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    x = generate(sample_fn)
    if DEV.type == "cuda": torch.cuda.synchronize()
    secs = (time.time() - t0) / len(EVAL_IDX)
    y = target_fn(EVAL_IDX)
    h1, h5 = hits(x, y)
    prec, rec = precision_recall_features(REAL_FEATS, clf_features(x), k=3)
    fid = fid_of(x) if cfg.n_eval >= 64 else float("nan")       # Inception statistics need a real sample size
    row = {"top1": h1.float().mean().item(), "top5": h5.float().mean().item(), "fid": fid,
           "precision": prec, "recall": rec, "seconds_per_sample": secs}
    METRICS["arms"][name] = row
    EVAL_SAMPLES[name] = (x[:8].cpu(), y[:8].cpu())
    print(f"{name:>14} {row['top1']:>7.3f} {row['top5']:>7.3f} {row['fid']:>8.2f} {prec:>6.3f} {rec:>6.3f}   {secs:>8.3f}")
if len(EVAL_IDX) < 2048:
    print("\nNOTE: fewer than 2048 images; FID is biased upward and accuracy is noisy. Not a result.")

## Manifest

In [ ]:
report = {
    "method": "TFG training-free guidance (arXiv:2409.15761), fine-tuned DeepFashion classifier",
    "modality": "images (DeepFashion 64x64, class target)",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "tfg_config": asdict(tfg_cfg),
    "guide_classifier": cfg.guide_clf,
    "eval_classifier": cfg.eval_clf,
    "classifier_real_val_accuracy": {k: {"top1": v[0], "top5": v[1]} for k, v in REAL_ACC.items()},
    "label_path": False,
    "accuracy": ACC,
    "n_eval_samples": N,
    "saturated_pixel_fraction": SATURATION,
    "sweep": sweep_rows,
    "seconds_per_sample": {"uncond": t_uncond, "label": t_label, "tfg": t_tfg},
    "tfg_cost_per_sample": {"denoiser_forward": tfg_stats.denoiser_evals,
                            "denoiser_backward": tfg_stats.denoiser_backward,
                            "objective_evals": tfg_stats.objective_evals},
    "plain_ddim_cost_per_sample": {"denoiser_forward": cfg.sample_steps},
    "base_best_val": min(h["val"] for h in base_hist),
    "checkpoint_meta": CKPT_META,
    "trainable_params": 0,
    "frozen_backbone": sum(p.numel() for p in base_frozen.parameters()),
    "nfe_sample_steps": cfg.sample_steps,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("METRICS", {}))
print(json.dumps(report, indent=2))
with open("manifest_tfg_fashion.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] Both classifiers' real-validation accuracy printed next to every generated-image accuracy;
      the scorer's is the ceiling. Both were fine-tuned on real training images only; say so
- [ ] `tfg_wrong_vs_target` is about as high as `tfg`: guidance follows its target
- [ ] `n_eval` at 2048 (the base notebooks' protocol); FID at that count is biased upward, say so,
      and read it with accuracy and precision / recall, not accuracy alone
- [ ] The base is the shared `checkpoints/diffusion_fashion/best.pt`; say so, or say it was retrained here
- [ ] Three **sampling** seeds on one checkpoint, mean +/- std, stated as such. Independently
      trained backbones would be a different, more expensive experiment; say which was done
- [ ] Guidance settings chosen on the validation sweep, then reported with fresh seeds
- [ ] Time per sample and the denoiser/objective call counts reported next to accuracy;
      TFG's cost is not a single NFE
- [ ] Saturation fraction reported; the objective is log p(y | clamp(x0))
- [ ] Described as TFG's `guide_step` **re-expressed** against our sampler with their CIFAR-10
      hyperparameters as starting settings (not tuned for DeepFashion) and `eta = 1`, on a
      null-label conditional model standing in for an unconditional one, with a fine-tuned
      classifier in place of a pretrained one